# 05. Agent Development Frameworks: Architecture vs Implementation

This course explores how modern agent frameworks encapsulate runtime mechanics. We compare four representative paradigms:
1. **Framework-Neutral Baseline (Raw Loop):** Application owns the loop, routing, state, and provider adapter.
2. **PydanticAI:** Framework owns the loop; specializes in type-safe outputs and dependency injection.
3. **LangGraph:** Framework models execution as a state machine graph; specializes in checkpointing and Human-in-the-Loop (HITL).
4. **OpenAI Agents SDK / Provider SDK:** Lightweight, provider-managed agent abstractions.

### Core Architectural Principles
- **Framework != Architecture:** The agent's capability boundary, state models, and safety invariants must be defined first.
- **The Grounding Invariant:** *No recommendation may rely on evidence the implementation did not retrieve.*
- **Persistence Semantics:** In-memory checkpointers (`MemorySaver`) demonstrate thread state resumption; durable persistence requires crash-safe backends (`SqliteSaver`, `PostgresSaver`).


In [ ]:
import json
import time
import os
import logging
from typing import Literal, List, Dict, Any, Optional, Set
from pydantic import BaseModel, Field, ConfigDict

# Centralized model configuration (Model/API capabilities evolve over time;
# official documentation at https://platform.openai.com/docs is the source of truth).
OPENAI_MODEL = 'gpt-4o-mini'
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Course05')
print(f'Environment initialized. Configured model: {OPENAI_MODEL}')


## Part 1 — Provider-Neutral Domain Models & Shared Tools

We define strict Pydantic inputs and outputs for our domain tools so that business logic remains completely decoupled from framework choices.


In [ ]:
import os
import sys

course_dir = os.path.join(
    os.getcwd(),
    "curriculum/beginner/05-agent-development-frameworks",
)
if course_dir not in sys.path:
    sys.path.insert(0, course_dir)

from policy import (
    HealthResult,
    IncidentRecommendation,
    DeploymentResult,
    AgentRunResult,
    ToolCall,
    ModelDecision
)
import json
from pydantic import BaseModel
from typing import Literal, List, Dict, Any


## Part 2 — The Framework-Neutral Baseline (Raw Loop)

To understand what frameworks automate, we build the loop manually.
### Grounded Multi-Step Trajectory
The agent executes a fully grounded trajectory:
1. `get_service_health(region='EU')` &rarr; observes degraded error rate
2. `get_recent_deployments(service='checkout')` &rarr; retrieves deployment `dep_eu_114`
3. Final recommendation synthesizing the evidence: EU checkout degradation began shortly after deployment `dep_eu_114`. The deployment is the leading hypothesis; verify deployment correlation/runbook criteria before rollback.

**Grounding Invariant:** The final recommendation only cites facts that were retrieved during the execution.


In [ ]:
class MockLLM:
    def __init__(self):
        self.turns = 0

    def chat(self, history: List[Dict[str, Any]]) -> ModelDecision:
        self.turns += 1
        if self.turns == 1:
            return ModelDecision(
                decision_summary='Investigate EU regional service health to confirm failure symptoms.',
                tool_calls=[ToolCall(id='tc_1', name='get_service_health', arguments_json='{"region": "EU"}')]
            )
        elif self.turns == 2:
            return ModelDecision(
                decision_summary='Service health is degraded. Retrieve recent checkout deployments to check for recent regressions.',
                tool_calls=[ToolCall(id='tc_2', name='get_recent_deployments', arguments_json='{"service": "checkout"}')]
            )
        else:
            return ModelDecision(
                decision_summary='Synthesize collected health and deployment evidence into final grounded recommendation.',
                final_answer=IncidentRecommendation(
                    summary='EU checkout degradation began shortly after deployment dep_eu_114.',
                    cited_evidence_ids=['health:EU', 'deploy:checkout:dep_eu_114'],
                    suspected_deployment_id='dep_eu_114',
                    attribution_strength='correlated',
                    recommended_action='consider_rollback'
                )
            )

def raw_agent_loop(query: str, max_steps: int = 5) -> AgentRunResult:
    llm = MockLLM()
    history = [{'role': 'user', 'content': query}]
    evidence_retrieved = []
    evidence_ids = []
    steps = 0
    tool_calls_count = 0
    print(f'[Raw Loop] Starting investigation: "{query}"')
    while steps < max_steps:
        steps += 1
        decision: ModelDecision = llm.chat(history)
        history.append({'role': 'model', 'decision': decision})
        print(f'Step {steps} | Action rationale: {decision.decision_summary}')
        
        if decision.final_answer:
            print(f'[Terminal] Final Answer: {decision.final_answer}')
            return AgentRunResult(
                recommendation=decision.final_answer,
                evidence_retrieved=evidence_retrieved,
                evidence_ids=evidence_ids,
                steps=steps,
                tool_calls=tool_calls_count
            )
        for tc in decision.tool_calls:
            tool_calls_count += 1
            if tc.name == 'get_service_health':
                res = get_service_health(HealthRequest.model_validate_json(tc.arguments_json))
            elif tc.name == 'get_recent_deployments':
                res = get_recent_deployments(DeploymentRequest.model_validate_json(tc.arguments_json))
            else:
                res = json.dumps({'error': f'Unknown tool {tc.name}'})
            if hasattr(res, 'evidence_id'):
                evidence_ids.append(res.evidence_id)
            evidence_retrieved.append(res)
            print(f'  [Observation] {res}')
            res_str = res.model_dump_json() if isinstance(res, BaseModel) else str(res)
            history.append({'role': 'tool', 'tool_id': tc.id, 'name': tc.name, 'content': res_str})
    raise TimeoutError('Step budget exceeded in raw loop.')

baseline_result = raw_agent_loop('Investigate EU checkout incident.')

# Verify Shared Grounding Invariant
import sys, os; sys.path.insert(0, os.path.join(os.getcwd(), 'curriculum/beginner/05-agent-development-frameworks')); from policy import verify_grounding
verify_grounding(baseline_result)


## Part 3 — PydanticAI: Typed Outputs & Dependency Injection

PydanticAI encapsulates the agent loop. Its defining architectural strengths are:
1. **Validated Typed Outputs (`output_type=FinalDecision`):** Validates model output against the declared structured type. Note: Typed output does NOT guarantee factual correctness, authorization, business correctness, or evidence grounding.
2. **Type-Safe Dependency Injection (`RunContext[AppDeps]`):** Injects application state (e.g. database connections, tenant credentials) safely into tool handlers.


In [ ]:
from dataclasses import dataclass

@dataclass
class AppDeps:
    environment: str

from pydantic_ai import Agent, RunContext, models
from pydantic_ai.models.test import TestModel

has_real_model = bool(os.getenv('OPENAI_API_KEY'))
if not has_real_model:
    models.ALLOW_MODEL_REQUESTS = False
    mock_result_dict = {
        "summary": "EU checkout degradation began shortly after deployment dep_eu_114.",
        "cited_evidence_ids": ["health:eu-central-1", "deploy:checkout:dep_eu_114"],
        "suspected_deployment_id": "dep_eu_114",
        "attribution_strength": "correlated",
        "recommended_action": "consider_rollback"
    }
    model = TestModel(call_tools=['check_health', 'check_deployments'], custom_output_args=mock_result_dict)
else:
    model = f'openai:{OPENAI_MODEL}'

# PydanticAI Agent with typed output and dependency injection
pydantic_agent = Agent(
    model,
    deps_type=AppDeps,
    output_type=IncidentRecommendation,
    system_prompt='You are an incident diagnostic assistant. Gather evidence with tools before recommending actions.'
)

@pydantic_agent.tool
def check_health(ctx: RunContext[AppDeps], region: str) -> str:
    logger.info(f'[PydanticAI Tool] Environment={ctx.deps.environment} checking region={region}')
    res = get_service_health(HealthRequest(region=region))
    return res.model_dump_json()

@pydantic_agent.tool
def check_deployments(ctx: RunContext[AppDeps], service: str) -> str:
    logger.info(f'[PydanticAI Tool] Environment={ctx.deps.environment} checking service={service}')
    res = get_recent_deployments(DeploymentRequest(service=service))
    return res.model_dump_json()

print('PydanticAI agent initialized successfully with output_type=IncidentRecommendation.')

deps = AppDeps(environment='production')
run_res = await pydantic_agent.run('Investigate EU checkout failures.', deps=deps)

# Access the validated structured data
print('\nPydanticAI Output:', run_res.output)
assert isinstance(run_res.output, IncidentRecommendation)
print('Successfully verified PydanticAI typed output invariant.')


## Part 4 — LangGraph: State Machine Graph & Checkpoint Semantics

LangGraph models the agent loop as an explicit directed graph with state checkpointing.

> [!IMPORTANT]
> **Persistence Clarification:** `MemorySaver` demonstrates in-memory checkpoint/resume semantics within a single process. > True crash-safe durability across process restarts requires a persistent checkpointer backend (such as `SqliteSaver` or `PostgresSaver`).


In [ ]:
try:
    from typing import Literal, Annotated
    from typing_extensions import TypedDict
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages
    from langgraph.checkpoint.memory import MemorySaver

    class GraphState(TypedDict):
        messages: Annotated[list, add_messages]
        human_approved: bool
        evidence_collected: list

    def agent_decide_node(state: GraphState):
        logger.info('[LangGraph Node: Agent] Evaluating gathered evidence.')
        return {
            'messages': [{'role': 'assistant', 'content': 'Diagnosis: EU checkout degradation began shortly after deployment dep_eu_114. The deployment is the leading hypothesis; verify deployment correlation/runbook criteria before rollback.'}],
            'evidence_collected': ['health:EU', 'deploy:checkout:dep_eu_114']
        }

    def human_review_node(state: GraphState):
        logger.info('[LangGraph Node: Review] Human on-call approved rollback.')
        return {'human_approved': True}

    builder = StateGraph(GraphState)
    builder.add_node('agent', agent_decide_node)
    builder.add_node('review', human_review_node)
    builder.add_edge(START, 'agent')
    builder.add_edge('agent', 'review')
    builder.add_edge('review', END)

    # In-memory checkpointer demonstrates checkpointing & interrupt/resume mechanics
    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer, interrupt_before=['review'])
    
    config = {'configurable': {'thread_id': 'incident_thread_101'}}
    
    print('\n--- 1. Graph Execution Pausing at Human Review Breakpoint ---')
    for event in graph.stream({'messages': [{'role': 'user', 'content': 'Investigate checkout'}], 'human_approved': False, 'evidence_collected': []}, config):
        pass
    
    checkpoint_state = graph.get_state(config)
    print('Thread is paused before node:', checkpoint_state.next)
    print('Current checkpoint values:', checkpoint_state.values)
    
    print('\n--- 2. Resuming Paused Thread After Human Approval ---')
    # Resumes execution from the exact checkpoint without re-running prior nodes
    for event in graph.stream(None, config):
        pass
    
    final_state = graph.get_state(config)
    print('Execution complete. Human approved status:', final_state.values['human_approved'])
    assert final_state.values['human_approved'] == True
except ImportError:
    print('langgraph is not installed. Graph architecture pattern defined above.')


## Part 5 — OpenAI Responses API & Agents SDK Patterns

For lightweight setups, developers often interact with provider SDKs directly or through minimal agent abstractions.
- **Raw Responses API:** Current `client.responses.create` and `client.responses.parse` multi-step tool loop.
- **OpenAI Agents SDK (`agents`):** Lightweight `Agent` + `Runner.run_sync` abstraction.


In [ ]:
# 1. OpenAI Raw Responses API Pattern
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected. Skipping live OpenAI calls.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    openai_tools = [
        {'type': 'function', 'name': 'get_service_health', 'description': 'Check region health', 'parameters': HealthRequest.model_json_schema()},
        {'type': 'function', 'name': 'get_recent_deployments', 'description': 'Check recent deployments', 'parameters': DeploymentRequest.model_json_schema()}
    ]

    conversation_input = [{'role': 'user', 'content': 'Investigate EU checkout failures.'}]

    print(f'\n--- Running Live OpenAI Responses API Loop ({OPENAI_MODEL}) ---')
    for step in range(1, 4):
        resp = client.responses.create(
            model=OPENAI_MODEL,
            instructions='You are an incident response assistant. Check service health and deployments before recommending actions.',
            input=conversation_input,
            tools=openai_tools
        )

        fn_calls = [item for item in resp.output if getattr(item, 'type', None) == 'function_call' or hasattr(item, 'call_id')]
        if fn_calls:
            tc = fn_calls[0]
            try:
                args = json.loads(tc.arguments) if isinstance(tc.arguments, str) else tc.arguments
            except Exception:
                args = {}
            print(f'Step {step} | Model requested tool: {tc.name}({args})')
            if tc.name == 'get_service_health':
                obs = get_service_health(HealthRequest.model_validate(args))
            elif tc.name == 'get_recent_deployments':
                obs = get_recent_deployments(DeploymentRequest.model_validate(args))
            else:
                obs = json.dumps({'error': 'Unknown tool'})
            
            print(f'  Observation: {obs}')
            call_id = getattr(tc, 'call_id', f'call_{step}')
            conversation_input.append({'type': 'function_call', 'call_id': call_id, 'name': tc.name, 'arguments': json.dumps(args)})
            conversation_input.append({'type': 'function_call_output', 'call_id': call_id, 'output': obs})
        else:
            print('\nFinal Model Answer:', resp.output_text)
            break

    # 2. Structured Output via Responses API
    print(f'\n--- Live Structured Output Parsing ({OPENAI_MODEL}) ---')
    try:
        structured_resp = client.responses.parse(
            model=OPENAI_MODEL,
            input=conversation_input,
            text_format=IncidentRecommendation
        )
        parsed_output = structured_resp.output_parsed
        print('Parsed IncidentRecommendation (Responses API):', parsed_output)
    except Exception:
        structured_resp = client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': 'Investigate EU checkout failures.'}],
            response_format=IncidentRecommendation
        )
        parsed_output: IncidentRecommendation = structured_resp.choices[0].message.parsed
        print('Parsed IncidentRecommendation (Chat fallback):', parsed_output)

# 3. OpenAI Agents SDK Pattern
try:
    from agents import Agent, Runner
    print('\n--- OpenAI Agents SDK Pattern ---')
    agent_instance = Agent(
        name='IncidentDiagnosticAgent',
        instructions='Diagnose incidents by gathering health and deployment evidence.',
        model=OPENAI_MODEL
    )
    print('OpenAI Agents SDK Agent initialized successfully:', agent_instance.name)
except ImportError:
    print('\nNote: openai-agents package not installed in the local environment. Architecture pattern defined above.')


## Part 6 — Objective Framework Comparison Matrix

We evaluate the four paradigms across critical production dimensions.


In [ ]:
import pandas as pd

framework_matrix = [
    {
        'Framework': 'Raw / Provider-Neutral',
        'Loop Ownership': 'Application Code',
        'Output Typing': 'Manual Pydantic Validation',
        'State Visibility': 'Full (Direct Python State)',
        'Persistence': 'Custom (Application DB)',
        'HITL': 'Native (Application Control Flow)',
        'Portability': 'High (Universal)',
        'Complexity': 'Low'
    },
    {
        'Framework': 'PydanticAI',
        'Loop Ownership': 'Framework Runtime',
        'Output Typing': 'Strict (output_type & Deps)',
        'State Visibility': 'Moderate (RunContext / Messages)',
        'Persistence': 'Optional (Message History)',
        'HITL': 'Manual / Message Resumption',
        'Portability': 'High (Multi-Model Support)',
        'Complexity': 'Medium'
    },
    {
        'Framework': 'LangGraph',
        'Loop Ownership': 'State Machine Graph',
        'Output Typing': 'Manual Node Schemas',
        'State Visibility': 'High (Graph State & Channels)',
        'Persistence': 'First-Class (Checkpointers)',
        'HITL': 'Built-in (interrupt_before/after)',
        'Portability': 'High (LangChain Ecosystem)',
        'Complexity': 'High'
    },
    {
        'Framework': 'OpenAI Agents SDK / Raw SDK',
        'Loop Ownership': 'SDK Runner / App Loop',
        'Output Typing': 'Built-in (.parse() / Pydantic)',
        'State Visibility': 'Moderate (Messages / Run Items)',
        'Persistence': 'External / Custom',
        'HITL': 'Manual Hand-offs',
        'Portability': 'Low-to-Medium (OpenAI Focused)',
        'Complexity': 'Low'
    }
]

df_comparison = pd.DataFrame(framework_matrix)
print('=== AGENT DEVELOPMENT FRAMEWORKS COMPARISON MATRIX ===')
print(df_comparison.to_string(index=False))
